# HackerNews Analytics Pro — v9 Temporal Intelligence

This notebook adds UTC-bucketed title-term evolution and transparent linear trend projections from repeated SQLite snapshots. These are descriptive baselines, not Transformer/LSTM forecasts. The supplied demo dataset is synthetic.

In [ ]:
from pathlib import Path
import pandas as pd
from hn_temporal_v9 import (build_topic_evolution, forecast_snapshot_engagement,
                                     backtest_snapshot_trends, export_v9_temporal)

ROOT = Path('..')
df = pd.read_json(ROOT / 'demo_stories.jsonl', lines=True)
df.head()

## 1. Topic evolution by UTC week
Title terms are counted within each period. This reveals lexical changes, not semantic topic meaning.

In [ ]:
evolution = build_topic_evolution(df, freq='W', top_n=12)
evolution_table = pd.DataFrame(evolution['periods'])
evolution_table.head()

In [ ]:
if evolution['terms']:
    display(evolution_table.set_index('period')[evolution['terms'][:6]].plot(figsize=(12, 5), title='Title-term frequency over time'))

## 2. Snapshot-based engagement projections
Run the live monitor for repeated observations before using these cells. At least three observations per story are required. The model fits a per-story least-squares line to score and comment counts.

In [ ]:
DB = ROOT / 'hn_monitor.sqlite'
if DB.exists():
    forecasts = forecast_snapshot_engagement(DB, horizon_hours=6)
    display(forecasts.head(20))
else:
    print('No snapshot database yet. Run: hn-analytics monitor --feed top --limit 100 --polls 4 --interval 300')

## 3. Chronological one-step backtest
The final observation per story is held out and compared with a persistence baseline. Small or sparse history can make this unavailable.

In [ ]:
if DB.exists():
    try:
        backtest_snapshot_trends(DB)
    except ValueError as exc:
        print(exc)

## 4. Export artifacts
The CLI equivalent is `hn-analytics temporal --input demo_stories.jsonl --output-dir reports`. Add `--db hn_monitor.sqlite` to include snapshot projections and backtest results.

In [ ]:
summary = export_v9_temporal(df, output_dir=ROOT / 'reports', db_path=DB if DB.exists() else None)
summary